In [ ]:
import json
import re
import html

try:
    with open(r'c:\Users\Flavio\Desktop\quizz CCNA1\questions.js', 'r', encoding='utf-8') as f:
        js_content = f.read()

    match = re.search(r'const\s+questions\s*=\s*(\[.*\]);?', js_content, re.DOTALL)
    if not match:
        raise ValueError("Could not find const questions = [...] in JS file.")
    questions_data = json.loads(match.group(1))

    with open(r'c:\Users\Flavio\Desktop\quizz CCNA1\page_source.html', 'r', encoding='utf-8') as f:
        html_content = f.read()

    def normalize_text(text):
        if text is None: return ""
        text = html.unescape(text)
        text = re.sub(r'<[^>]+>', '', text)
        text = re.sub(r'\W+', '', text).lower()
        return text

    def parse_html_to_map(content):
        q_map = {}
        # Split by something that delineates questions.
        # Often <p><strong>Question...</strong></p> followed by <ul>...</ul>
        # We can split by <ul> blocks. The text preceding is the question.
        # However, there may be images.
        
        # Strategy: Find all <ul>...</ul> blocks.
        # Extract <li> items. Check for correctness.
        # Look at text immediately preceding the <ul> block.
        
        content = content.replace('\n', ' ')
        parts = re.split(r'(<ul.*?>.*?</ul>)', content, flags=re.IGNORECASE)
        
        last_text = ""
        for i in range(len(parts)):
            part = parts[i]
            if part.strip().lower().startswith('<ul'):
                # Process options
                options = re.findall(r'<li(.*?)>(.*?)</li>', part, flags=re.IGNORECASE)
                correct_opts = []
                for attrs, opt_text in options:
                    is_correct = False
                    if 'correct_answer' in attrs or 'color: #ff0000' in attrs or 'color: red' in attrs:
                        is_correct = True
                    if 'color: #ff0000' in opt_text:
                        is_correct = True
                    
                    if is_correct:
                        correct_opts.append(normalize_text(opt_text))
                
                if correct_opts:
                    # Find question text in last_text
                    strongs = re.findall(r'<strong>(.*?)</strong>', last_text, flags=re.IGNORECASE)
                    if strongs:
                        # Take the last strong tag
                        q_raw = strongs[-1]
                        # Remove leading number "12. "
                        q_raw = re.sub(r'^\d+\.\s*', '', q_raw)
                        q_key = normalize_text(q_raw)
                        if q_key:
                            q_map[q_key] = correct_opts
            
            else:
                last_text = part
        return q_map

    html_qa_map = parse_html_to_map(html_content)
    
    updates = []
    
    for idx, q_item in enumerate(questions_data):
        if q_item.get('answer') is None:
            q_text = q_item.get('question', '')
            q_key = normalize_text(q_text)
            
            correct_opts = html_qa_map.get(q_key)
            
            if not correct_opts:
                # Try fuzzy matching or stripping parens
                q_text_stripped = re.sub(r'\s*\(.*?\)', '', q_text)
                key_stripped = normalize_text(q_text_stripped)
                correct_opts = html_qa_map.get(key_stripped)
            
            if not correct_opts:
                # Try partial match within keys
                # This is slow O(N*M) but N is small
                for k, v in html_qa_map.items():
                    if len(k) > 20 and (k in q_key or q_key in k):
                        correct_opts = v
                        break
            
            if correct_opts:
                found_indices = []
                for i, opt in enumerate(q_item.get('options', [])):
                    norm_opt = normalize_text(opt)
                    
                    # Exact match
                    if norm_opt in correct_opts:
                        found_indices.append(i)
                    else:
                        # Partial match
                        for c_opt in correct_opts:
                            if len(norm_opt) > 10 and (c_opt in norm_opt or norm_opt in c_opt):
                                found_indices.append(i)
                                break
                
                found_indices = sorted(list(set(found_indices)))
                if found_indices:
                    update_entry = {"index": idx, "answer": found_indices[0] if len(found_indices) == 1 else found_indices}
                    updates.append(update_entry)

    print(json.dumps(updates))

except Exception as e:
    print(f"Error: {e}")

In [ ]:

import json
import re
import html

# Files
QUESTIONS_FILE = r"c:\Users\Flavio\Desktop\quizz CCNA1\questions.js"
HTML_FILE = r"c:\Users\Flavio\Desktop\quizz CCNA1\page_source.html"

def load_questions():
    with open(QUESTIONS_FILE, 'r', encoding='utf-8') as f:
        content = f.read()
        content = re.sub(r'^\s*const\s+questions\s*=\s*', '', content)
        content = re.sub(r';\s*$', '', content)
        return json.loads(content)

def load_html():
    with open(HTML_FILE, 'r', encoding='utf-8') as f:
        return f.read()

def normalize_text(text):
    if not text: return ""
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'^\d+\.\s*', '', text) 
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text

def check_missing():
    try:
        questions = load_questions()
        html_content = load_html()
        
        normalized_existing = set()
        for q in questions:
            normalized_existing.add(normalize_text(q['question']))
        
        matches = list(re.finditer(r'<strong>(\d+)\.\s*(.*?)</strong>', html_content))
        
        missing_count = 0
        for m in matches:
            num = m.group(1)
            txt = m.group(2)
            norm = normalize_text(txt)
            
            found = False
            for ex in normalized_existing:
                if norm == ex or (len(norm) > 20 and norm in ex) or (len(ex) > 20 and ex in norm):
                    found = True
                    break
            
            if not found:
                print(f"Missing Q{num}")
                missing_count += 1

        print(f"Total Missing: {missing_count}")
        print(f"Total in JS: {len(questions)}")
        
    except Exception as e:
        print(f"Error: {e}")

check_missing()
